### Retrieving the Dataset

In [5]:
import ee

# To run the code, it is necessary to register and authenticate in Google Earth Engine
ee.Authenticate()
ee.Initialize(project='time-series-462612')

# --------------------------------------------------
# 1. INPUT ASSETS
# --------------------------------------------------
train_p1 = ee.FeatureCollection("projects/ee-babakghassemi9/assets/LU22_S1S2MYA_CentroPolyPointsFull_training75_part_1")
train_p2 = ee.FeatureCollection("projects/ee-babakghassemi9/assets/LU22_S1S2MYA_CentroPolyPointsFull_training75_part_2")
train_p3 = ee.FeatureCollection("projects/ee-babakghassemi9/assets/LU22_S1S2MYA_CentroPolyPointsFull_training75_part_3")
train_p4 = ee.FeatureCollection("projects/ee-babakghassemi9/assets/LU22_S1S2MYA_CentroPolyPointsFull_training75_part_4")

f_name1 = ee.FeatureCollection("projects/ee-babakghassemi9/assets/LU22_S1S2MYA_important_features2")

train_all = (train_p1
             .merge(train_p2)
             .merge(train_p3)
             .merge(train_p4))

labels = (
    train_all
    .aggregate_array("Label_clas")
    .distinct()
    .sort()
    .getInfo()
)

print("Unique Label_clas values:", labels)
print(len(labels))

KeyboardInterrupt: Interrupted by user

In [ ]:
label_prop = "Label_clas"

feat_select = (
    ee.FeatureCollection(f_name1)
    .aggregate_array('0')   # column holding feature names
    .distinct()
)

feat_select_py = feat_select.getInfo()   # ← critical step

selectors = list(feat_select_py)  # make a copy

if label_prop not in selectors:
    selectors.append(label_prop)

train_tabular = (
    train_all
    .select(selectors)
    .map(lambda f: f.setGeometry(None))
)

train_tabular_minor = ee.FeatureCollection(train_tabular) \
    .filter(ee.Filter.eq('Lbl_cls_major', 1)) \
    .select(feat_select.cat(ee.List(['Label_clas'])))


task = ee.batch.Export.table.toDrive(
    collection=train_tabular,
    description="L22_tabular_csv",
    fileFormat="CSV",
    selectors=selectors
)
task.start()


task = ee.batch.Export.table.toDrive(
    collection=train_tabular_minor,
    description="L22_tabular-minor_csv",
    fileFormat="CSV",
    selectors=selectors
)
task.start()

### Dataset

In [4]:
import pandas as pd
df = pd.read_csv("datasets/LU22_tabular.csv")
df

,0_2022_B3,0_2022_B4,0_2022_B5,0_2022_BLFEI,0_2022_LAI,0_2022_LAIg,0_2022_LCCI,0_2022_MSAVI,0_2022_NDVI,0_2022_SAVI,...,6_VV,6_VV-VH,7_VH,DPSVIm_p5,NDPI_p50,RVI_p5,VH_p5,VH_p50,VV/VH_p50,Label_clas
0,843.75,883.333333,929.0,-0.118480,0.978622,1.537698,1.741266,0.212417,0.423573,0.236555,...,0.115004,0.073613,0.027390,0.001123,0.541142,0.400348,0.007754,0.022213,3.414194,0.0
1,843.00,1012.000000,1079.5,-0.247500,0.663848,0.874161,1.287353,0.179741,0.355224,0.206760,...,0.243982,0.227067,0.033261,0.002960,0.752911,0.106674,0.009098,0.021289,7.094262,0.0
2,1332.00,1534.000000,1978.0,-0.204278,0.324716,0.266992,1.125175,0.095925,0.190886,0.109272,...,0.103763,0.073865,0.046867,0.002759,0.532950,0.334808,0.019387,0.045704,3.282224,0.0
3,522.00,558.500000,812.5,-0.350248,0.201110,1.455451,1.676414,0.073488,0.310730,0.097483,...,0.043246,0.030165,0.011817,0.000422,0.590604,0.224218,0.002996,0.012626,3.850154,0.0
4,816.00,959.000000,1042.0,-0.300304,0.351574,0.849538,1.260390,0.106926,0.254429,0.129364,...,0.074618,0.064405,0.013658,0.000814,0.599835,0.318086,0.006434,0.016966,4.006589,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86826,452.50,533.500000,653.0,-0.414838,0.345434,1.349546,1.397779,0.106885,0.356388,0.134119,...,0.040304,0.023955,0.012031,0.000446,0.493432,0.339069,0.004765,0.015061,2.948141,24.0
86827,474.00,552.000000,709.5,-0.397990,0.385781,1.255766,1.465444,0.116721,0.367725,0.144453,...,0.035737,0.022999,0.017686,0.000315,0.525086,0.270285,0.002501,0.013924,3.185776,24.0
86828,433.00,524.500000,560.5,-0.387854,0.372217,1.490952,1.541165,0.112493,0.386439,0.142014,...,0.034708,0.021457,0.014751,0.000310,0.479512,0.364908,0.003263,0.014028,2.836476,24.0
86829,550.00,623.000000,902.0,-0.336877,0.260788,0.601186,1.194049,0.089348,0.259740,0.116107,...,0.066215,0.046820,0.022128,0.000920,0.597547,0.217218,0.003333,0.016231,4.131786,24.0


In [ ]:
df = pd.read_csv('./datasets/LU22_minor_tabular.csv')
print(df['Label_clas'].unique())  # What are the actual values?
print(df['Label_clas'].min(), df['Label_clas'].max())  # Range?

[ 5.  1.  2.  3.  4.  6.  8. 10. 11. 12. 14. 15. 17. 18. 19.  9. 13. 16.
  7.]
1.0 19.0


In [ ]:
df = pd.read_csv('./datasets/LU22_minor_tabular.csv')

# See original labels
print("Original labels:", sorted(df['Label_clas'].unique()))

# Remap
original_labels = sorted(df['Label_clas'].unique())
label_map = {old: new for new, old in enumerate(original_labels)}

print("\nMapping:")
for old, new in label_map.items():
    print(f"  {old} -> {new}")

# Apply
df['Label_clas'] = df['Label_clas'].map(label_map)

# Save
df.to_csv('./datasets/LU22_minor_tabular.csv', index=False)
print("\nSaved to LU22_minor_tabular.csv")

Original labels: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0), np.float64(11.0), np.float64(12.0), np.float64(13.0), np.float64(14.0), np.float64(15.0), np.float64(16.0), np.float64(17.0), np.float64(18.0), np.float64(19.0)]

Mapping:
  1.0 -> 0
  2.0 -> 1
  3.0 -> 2
  4.0 -> 3
  5.0 -> 4
  6.0 -> 5
  7.0 -> 6
  8.0 -> 7
  9.0 -> 8
  10.0 -> 9
  11.0 -> 10
  12.0 -> 11
  13.0 -> 12
  14.0 -> 13
  15.0 -> 14
  16.0 -> 15
  17.0 -> 16
  18.0 -> 17
  19.0 -> 18

Saved to LU22_minor_tabular.csv
